In [2]:
import min_features, daily_return
import importlib
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.inspection import permutation_importance
import warnings
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

importlib.reload(min_features)
importlib.reload(daily_return)

df_min = min_features.min_features()
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 

df_main = pd.merge(df_min, df_daily, how='inner', on='Date')
df_main = df_main.sort_values(by='Date', ascending=False)

return_cols = df_main.columns[df_main.columns.str.contains("Return_")].to_list()
daily_cols = [
    c for c in df_daily.iloc[:, 1:].columns
    if "return" not in c.lower()
]
close_cols = df_min.columns[(df_min.columns.str.contains("close_")) | (df_min.columns.str.contains("post_")) | (df_min.columns.str.contains("overnight_"))].to_list()
min_cols = (
    df_min
    .loc[:, ~df_min.columns.isin(close_cols)]  # drop close_ columns
    .iloc[:, 1:]                               # drop first column
    .columns
    .to_list()
)
#results_baseline = pd.read_csv("baseline_performance_1-3-5-10.csv")


In [ ]:
# -----------------------------
# Models
# -----------------------------
models = {
    "xgboost": XGBClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "random_forest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
}

# -----------------------------
# Helpers
# -----------------------------
def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else 0 #int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    #y_all = _to_binary(dfw[target_col].to_numpy())
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
            f"Run {k+1}/{runs} | "
            f"Train: {dates[train_start]} → {dates[train_end-1]} | "
            f"Test: {dates[test_start]} → {dates[test_end-1]} | "
            f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test  = X_all[test_start:test_end]
        y_test  = y_all[test_start:test_end]

        dist = _compute_dist(y_test)
        single_class_test = (np.unique(y_test).size < 2)

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)

            preds = m.predict(X_test)

            """
            # probabilities if available (for confidence metrics)
            proba = None
            if hasattr(m, "predict_proba"):
                proba = m.predict_proba(X_test)[:, 1]
            elif hasattr(m, "decision_function"):
                s = m.decision_function(X_test)
                # squash to (0,1) so confidence metrics work consistently
                proba = 1.0 / (1.0 + np.exp(-s))

            # confidence/coverage metrics (optional but useful)
            topk_acc = np.nan
            topk_cov = np.nan
            if proba is not None and len(proba) > 0:
                conf = np.abs(proba - 0.5)
                # top 40% by confidence (with 5 samples, this is ~2 samples)
                q = np.quantile(conf, 0.60)
                sel = conf >= q
                topk_cov = float(sel.mean())
                topk_acc = float((preds[sel] == y_test[sel]).mean()) if sel.any() else np.nan
            """
            rows.append({
                "run": k + 1,
                "model": model_name,
                "test_days": test_days,

                # core metrics
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "acc": float(accuracy_score(y_test, preds)),
                "sign_acc": 2 * float(accuracy_score(y_test, preds)) - 1,
                "mcc": float(matthews_corrcoef(y_test, preds)),

                # only meaningful if test has both classes
                "f1": np.nan if single_class_test else float(f1_score(y_test, preds, zero_division=0)),
                "precision": np.nan if single_class_test else float(precision_score(y_test, preds, zero_division=0)),
                "recall": np.nan if single_class_test else float(recall_score(y_test, preds, zero_division=0)),

                # confidence-conditioned performance (if proba/decision_function exists)
                #"top40_acc": topk_acc,
                #"top40_cov": topk_cov,

                **dist,

                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "train_years": train_years,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [2, 5, 10]#, 3, 5]  # add 3,5,etc later
train_years_grid = [4, 6]#[3, 5, 7]  # could be [3,4,5,6]
days_assessed = 228
test_days = [1, 2, 3, 4]
#runs = 75

results= []
results_df = pd.DataFrame()

for test_day in test_days:
    runs = days_assessed / test_day
    for feature_cols, feat_name in zip(column_sets, names):
        for r in returns:
            print(f"{r} | {feat_name}")
            target_col = f"Return_{r}"

            for train_years in train_years_grid:
                df_scores = walkback_runs(
                    df=df_main,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    date_col="Date",
                    train_years=train_years,
                    test_days=test_day,
                    step_days=test_day,
                    runs=runs,
                    horizon_days=r,
                    purge_days=None,   # purge = horizon (safe default)
                    fill_inf=0.0,
                )

                df_scores["feature_set"] = feat_name
                df_scores["horizon"] = r

                results.append(df_scores)

results_df = pd.concat(results, ignore_index=True)

1 | daily
Run 1/75 | Train: 2023-01-03 → 2025-12-16 | Test: 2025-12-17 → 2025-12-19 | Train_n=735 | Test_n=3
Run 2/75 | Train: 2022-12-23 → 2025-12-09 | Test: 2025-12-10 → 2025-12-12 | Train_n=735 | Test_n=3
Run 3/75 | Train: 2022-12-16 → 2025-12-02 | Test: 2025-12-03 → 2025-12-05 | Train_n=735 | Test_n=3
Run 4/75 | Train: 2022-12-09 → 2025-11-21 | Test: 2025-11-24 → 2025-11-26 | Train_n=735 | Test_n=3
Run 5/75 | Train: 2022-12-02 → 2025-11-14 | Test: 2025-11-17 → 2025-11-19 | Train_n=735 | Test_n=3
Run 6/75 | Train: 2022-11-23 → 2025-11-07 | Test: 2025-11-10 → 2025-11-12 | Train_n=735 | Test_n=3
Run 7/75 | Train: 2022-11-16 → 2025-10-31 | Test: 2025-11-03 → 2025-11-05 | Train_n=735 | Test_n=3
Run 8/75 | Train: 2022-11-09 → 2025-10-24 | Test: 2025-10-27 → 2025-10-29 | Train_n=735 | Test_n=3
Run 9/75 | Train: 2022-11-02 → 2025-10-17 | Test: 2025-10-20 → 2025-10-22 | Train_n=735 | Test_n=3
Run 10/75 | Train: 2022-10-26 → 2025-10-10 | Test: 2025-10-13 → 2025-10-15 | Train_n=735 | Test_n=3

In [21]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
idx  = ["feature_set", "horizon", "train_years", "model", 'test_days']

dfs = [pd.read_csv("master_run_results.csv")]
final_df = None
metric = 'acc' #'signed_acc, acc

for df in dfs:

    df = df[df['test_days'] == 1].copy()

    # bucket to exact 5-day bins
    #df["test_pos_frac"] = ((df["test_pos_frac"] * 5).round() / 5).clip(0, 1)

    # overall (distribution-agnostic)
    overall = (
        df.groupby(idx, as_index=False)
          .agg(m_all=(metric, "mean"), n_all=("run", "count"))
    )

    # by-bin
    g = (
        df.groupby(idx + ["test_pos_frac"], as_index=False)
          .agg(m=(metric, "mean"), n=("run", "count"))
    )

    wide = (
        g.pivot(index=idx, columns="test_pos_frac", values=["m", "n"])
         .reindex(columns=bins, level=1)
    )
    wide.columns = [f"{metric}_{frac:g}" for metric, frac in wide.columns]
    wide = wide.reset_index()

    # merge overall into wide
    wide = wide.merge(overall, on=idx, how="left")

    #wide = wide[column_order].round(3)

    if final_df is None:
        final_df = wide.copy()
    else:
        final_df = pd.concat([final_df, wide.copy()], ignore_index=True)

final_df['bal_acc'] = (final_df['m_0'] + final_df['m_1']) / 2
column_order = ['horizon', 'feature_set', 'train_years', 'model', 'test_days',
                'n_1', 'n_0', 'm_1', 'm_0', 'm_all', 'n_all', 'bal_acc']
final_df[column_order][final_df['horizon'] >= 2].sort_values(by=['horizon', 'bal_acc'], ascending=False).round(2)

,horizon,feature_set,train_years,model,test_days,n_1,n_0,m_1,m_0,m_all,n_all,bal_acc
8,10,daily,4,random_forest,1,148.0,80.0,0.91,0.80,0.87,228,0.86
10,10,daily,6,random_forest,1,148.0,80.0,0.91,0.78,0.86,228,0.84
11,10,daily,6,xgboost,1,148.0,80.0,0.91,0.78,0.86,228,0.84
9,10,daily,4,xgboost,1,148.0,80.0,0.93,0.74,0.86,228,0.83
20,10,daily+minute,4,random_forest,1,148.0,80.0,0.89,0.75,0.84,228,0.82
22,10,daily+minute,6,random_forest,1,148.0,80.0,0.92,0.71,0.85,228,0.82
21,10,daily+minute,4,xgboost,1,148.0,80.0,0.89,0.66,0.81,228,0.78
23,10,daily+minute,6,xgboost,1,148.0,80.0,0.91,0.62,0.81,228,0.77
35,10,minute,6,xgboost,1,148.0,80.0,0.80,0.24,0.61,228,0.52
34,10,minute,6,random_forest,1,148.0,80.0,0.97,0.08,0.65,228,0.52
